# Statistical Validation and Sensitivity Analysis

## 1. Purpose

This notebook validates whether employment status is associated with Low-Intensity Placement among admissions with co-occurring mental health needs.
It includes chi-square testing, adjusted logistic regression, and stratified sensitivity checks by age, wait time, and state.

## 2. Load Analysis Dataset from Fabric Lakehouse

In [ ]:
df = spark.table("silver_tedsa_admissions_cleaned")

display(df.limit(10))

StatementMeta(, fcd4a56d-2e8c-4f15-b96c-3d474d1f87f2, 4, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 2c4e1ca4-b733-412a-a079-f5b55337be4f)

## 3. Create Analysis Sample

In [ ]:
from pyspark.sql import functions as F

# Load source table from Fabric Lakehouse
df = spark.table("silver_tedsa_admissions_cleaned")

# Check source fields
print("Columns in silver_tedsa_admissions_cleaned:")
print(df.columns)

# Build analysis sample based on actual TEDS-A / Silver table fields
# Source-field mapping:
# PSYPROB = co-occurring mental health flag
# EMPLOY / EMPLOY_Label = employment status
# SEX = sex field, not GENDER
# STFIPS = state FIPS field, not STATE
# Treatment_Intensity = existing treatment intensity field in Silver table

analysis_df = (
    df
    .filter(F.col("PSYPROB") == 1)
    .filter(F.col("EMPLOY").isin(1, 2, 3, 4))
    .filter(F.col("Treatment_Intensity").isNotNull())
    .withColumn(
        "Employment_Status",
        F.when(F.col("EMPLOY") == 1, "Full-time employed")
         .when(F.col("EMPLOY") == 2, "Part-time employed")
         .when(F.col("EMPLOY") == 3, "Unemployed")
         .when(F.col("EMPLOY") == 4, "Not in labor force")
    )
    .withColumn(
        "Low_Intensity_Placement",
        F.when(F.lower(F.col("Treatment_Intensity")).contains("low"), F.lit(1))
         .otherwise(F.lit(0))
    )
    .select(
        "Low_Intensity_Placement",
        "Treatment_Intensity",
        "Employment_Status",
        "EMPLOY",
        "AGE",
        "SEX",
        "RACE",
        "STFIPS",
        "DAYWAIT"
    )
    .dropna()
)

display(analysis_df.limit(10))

StatementMeta(, 782d0187-be47-4ab6-8924-3e81cedd39aa, 3, Finished, Available, Finished, False)

Columns in silver_tedsa_admissions_cleaned:
['ADMYR', 'CASEID', 'STFIPS', 'EDUC', 'MARSTAT', 'SERVICES', 'DETCRIM', 'NOPRIOR', 'PSOURCE', 'ARRESTS', 'EMPLOY', 'METHUSE', 'PSYPROB', 'PREG', 'SEX', 'VET', 'LIVARAG', 'DAYWAIT', 'DSMCRIT', 'AGE', 'RACE', 'ETHNIC', 'DETNLF', 'PRIMINC', 'SUB1', 'SUB2', 'SUB3', 'ROUTE1', 'ROUTE2', 'ROUTE3', 'FREQ1', 'FREQ2', 'FREQ3', 'FRSTUSE1', 'FRSTUSE2', 'FRSTUSE3', 'HLTHINS', 'PRIMPAY', 'FREQ_ATND_SELF_HELP', 'ALCFLG', 'COKEFLG', 'MARFLG', 'HERFLG', 'METHFLG', 'OPSYNFLG', 'PCPFLG', 'HALLFLG', 'MTHAMFLG', 'AMPHFLG', 'STIMFLG', 'BENZFLG', 'TRNQFLG', 'BARBFLG', 'SEDHPFLG', 'INHFLG', 'OTCFLG', 'OTHERFLG', 'DIVISION', 'REGION', 'IDU', 'ALCDRUG', 'CBSA2020', 'PSYPROB_Label', 'EMPLOY_Label', 'Wait_Time_Tier', 'Risk_Profile', 'Treatment_Intensity']


SynapseWidget(Synapse.DataFrame, ce0aff19-5fc0-4117-a11b-0678de71d3e0)

## 4. Check Sample Size

In [ ]:
analysis_df.count()

StatementMeta(, 782d0187-be47-4ab6-8924-3e81cedd39aa, 4, Finished, Available, Finished, False)

538172

## 5. Convert Spark DataFrame to Pandas

In [ ]:
pdf = analysis_df.toPandas()

pdf.head()

StatementMeta(, 782d0187-be47-4ab6-8924-3e81cedd39aa, 6, Finished, Available, Finished, False)

,Low_Intensity_Placement,Treatment_Intensity,Employment_Status,EMPLOY,AGE,SEX,RACE,STFIPS,DAYWAIT
0,0,High Intensity,Unemployed,3,10,1,5,9,2
1,1,Low Intensity,Part-time employed,2,10,2,4,9,0
2,0,Medium Intensity,Unemployed,3,4,1,5,9,-9
3,0,Medium Intensity,Unemployed,3,4,1,5,9,-9
4,0,Medium Intensity,Unemployed,3,4,1,5,9,-9


## 6. Chi-Square Test: Employment Status vs Low-Intensity Placement

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency

chi_pdf = (
    analysis_df
    .groupBy("Employment_Status", "Low_Intensity_Placement")
    .count()
    .toPandas()
)

contingency_table = chi_pdf.pivot(
    index="Employment_Status",
    columns="Low_Intensity_Placement",
    values="count"
).fillna(0)

chi2, p_value, dof, expected = chi2_contingency(contingency_table)
n = contingency_table.to_numpy().sum()
min_dimension = min(contingency_table.shape) - 1
cramers_v = np.sqrt(chi2 / (n * min_dimension))

print("Contingency Table:")
print(contingency_table)

print("\nChi-square statistic:", chi2)
print("Degrees of freedom:", dof)
print("P-value:", p_value)
print("Cramer's V:", cramers_v)

StatementMeta(, 782d0187-be47-4ab6-8924-3e81cedd39aa, 8, Finished, Available, Finished, False)

Contingency Table:
Low_Intensity_Placement       0      1
Employment_Status                
Full-time employed   31517  59889
Not in labor force  100368  95283
Part-time employed   10640  23360
Unemployed          126013  91102

Chi-square statistic: 19308.835366538267
Degrees of freedom: 3
P-value: 0.0


## 7. Logistic Regression: Adjusted Association with Low-Intensity Placement

In [ ]:
pdf = analysis_df.toPandas()

pdf.head()

StatementMeta(, 782d0187-be47-4ab6-8924-3e81cedd39aa, 10, Finished, Available, Finished, False)

,Low_Intensity_Placement,Treatment_Intensity,Employment_Status,EMPLOY,AGE,SEX,RACE,STFIPS,DAYWAIT
0,0,High Intensity,Unemployed,3,10,1,5,9,2
1,1,Low Intensity,Part-time employed,2,10,2,4,9,0
2,0,Medium Intensity,Unemployed,3,4,1,5,9,-9
3,0,Medium Intensity,Unemployed,3,4,1,5,9,-9
4,0,Medium Intensity,Unemployed,3,4,1,5,9,-9


In [ ]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf

model = smf.logit(
    "Low_Intensity_Placement ~ C(Employment_Status) + C(AGE) + C(SEX) + C(RACE) + C(STFIPS) + C(DAYWAIT)",
    data=pdf
).fit()

print(model.summary())

StatementMeta(, 782d0187-be47-4ab6-8924-3e81cedd39aa, 12, Finished, Available, Finished, False)

Optimization terminated successfully.
         Current function value: 0.601293
         Iterations 7


                           Logit Regression Results                           
Dep. Variable:     Low_Intensity_Placement   No. Observations:               538172
Model:                          Logit   Df Residuals:                   538099
Method:                           MLE   Df Model:                           72
Date:                Fri, 26 Jun 2026   Pseudo R-squ.:                  0.1325
Time:                        21:45:25   Log-Likelihood:            -3.2360e+05
converged:                       True   LL-Null:                   -3.7303e+05
Covariance Type:            nonrobust   LLR p-value:                     0.000
                                                 coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------------------------------
Intercept                                      2.6572      0.123     21.521      0.000       2.415       2.899
C(Employment_Status)[T.Not in 

## 8. Convert Logistic Regression Coefficients to Odds Ratios

In [ ]:
odds_ratios = pd.DataFrame({
    "variable": model.params.index,
    "odds_ratio": np.exp(model.params.values),
    "coef": model.params.values,
    "p_value": model.pvalues.values
})

odds_ratios

StatementMeta(, 782d0187-be47-4ab6-8924-3e81cedd39aa, 13, Finished, Available, Finished, False)

,variable,odds_ratio,coef,p_value
0,Intercept,14.256871,2.657239,9.801893e-103
1,C(Employment_Status)[T.Not in labor force],0.411077,-0.888974,0.000000e+00
2,C(Employment_Status)[T.Part-time employed],1.066245,0.064143,1.164513e-05
3,C(Employment_Status)[T.Unemployed],0.402622,-0.909758,0.000000e+00
4,C(AGE)[T.2],0.694687,-0.364293,3.968015e-11
...,...,...,...,...
68,C(DAYWAIT)[T.0],0.938832,-0.063119,8.369619e-07
69,C(DAYWAIT)[T.1],0.899561,-0.105849,3.275646e-12
70,C(DAYWAIT)[T.2],1.426846,0.355466,5.468600e-66
71,C(DAYWAIT)[T.3],1.536973,0.429815,2.430786e-81


## 9. Sensitivity Analysis by Age

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Calculate low-intensity placement rate by AGE and Employment_Status
age_sensitivity = (
    analysis_df
    .groupBy("AGE", "Employment_Status")
    .agg(
        F.count("*").alias("admissions"),
        F.sum("Low_Intensity_Placement").alias("low_intensity_admissions")
    )
    .withColumn(
        "low_intensity_placement_rate",
        F.col("low_intensity_admissions") / F.col("admissions")
    )
)

# Rank employment groups within each age group by low-intensity placement rate
age_window = Window.partitionBy("AGE").orderBy(F.col("low_intensity_placement_rate").desc())

age_sensitivity_ranked = (
    age_sensitivity
    .withColumn("rank_within_age", F.row_number().over(age_window))
    .orderBy("AGE", "rank_within_age")
)

display(age_sensitivity_ranked)

StatementMeta(, 782d0187-be47-4ab6-8924-3e81cedd39aa, 15, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 9d83c8cb-4dd2-4557-838a-96c8c57e3d39)

## 9.1 Age Sensitivity Summary

In [ ]:
age_top_group_summary = (
    age_sensitivity_ranked
    .filter(F.col("rank_within_age") == 1)
    .groupBy("Employment_Status")
    .agg(
        F.count("*").alias("age_groups_ranked_highest")
    )
    .orderBy(F.col("age_groups_ranked_highest").desc())
)

display(age_top_group_summary)

StatementMeta(, 782d0187-be47-4ab6-8924-3e81cedd39aa, 16, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 8dd50460-d467-4915-b802-e4dce8403d6c)

## 10. Sensitivity Analysis by Wait Time

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Create a readable wait-time grouping.
# DAYWAIT is coded as admission wait time category in the source data.
wait_sensitivity_base = (
    analysis_df
    .withColumn(
        "Wait_Time_Group",
        F.when(F.col("DAYWAIT") == 0, "0 days")
         .when(F.col("DAYWAIT") == 1, "1-7 days")
         .when(F.col("DAYWAIT") == 2, "8-14 days")
         .when(F.col("DAYWAIT") == 3, "15-30 days")
         .when(F.col("DAYWAIT") == 4, "31+ days")
         .otherwise("Unknown / not collected")
    )
    .filter(F.col("Wait_Time_Group") != "Unknown / not collected")
)

# Calculate low-intensity placement rate by wait-time group and employment status
wait_time_sensitivity = (
    wait_sensitivity_base
    .groupBy("Wait_Time_Group", "Employment_Status")
    .agg(
        F.count("*").alias("admissions"),
        F.sum("Low_Intensity_Placement").alias("low_intensity_admissions")
    )
    .withColumn(
        "low_intensity_placement_rate",
        F.col("low_intensity_admissions") / F.col("admissions")
    )
)

# Rank employment groups within each wait-time group by low-intensity placement rate
wait_window = Window.partitionBy("Wait_Time_Group").orderBy(F.col("low_intensity_placement_rate").desc())

wait_time_sensitivity_ranked = (
    wait_time_sensitivity
    .withColumn("rank_within_wait_time", F.row_number().over(wait_window))
    .orderBy("Wait_Time_Group", "rank_within_wait_time")
)

display(wait_time_sensitivity_ranked)

StatementMeta(, 782d0187-be47-4ab6-8924-3e81cedd39aa, 17, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 3fce3580-844f-408a-8f4b-1a3c13fd2de3)

## 10.1 Wait-Time Sensitivity Summary

In [ ]:
wait_time_top_group_summary = (
    wait_time_sensitivity_ranked
    .filter(F.col("rank_within_wait_time") == 1)
    .groupBy("Employment_Status")
    .agg(
        F.count("*").alias("wait_time_groups_ranked_highest")
    )
    .orderBy(F.col("wait_time_groups_ranked_highest").desc())
)

display(wait_time_top_group_summary)

StatementMeta(, 782d0187-be47-4ab6-8924-3e81cedd39aa, 18, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 81f2aa5e-1737-488f-9707-3b1fb910d9e2)

## 11. Sensitivity Analysis by State

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# State-level sensitivity check
# Use a minimum sample threshold to avoid over-interpreting small state-employment cells.
MIN_STATE_EMPLOYMENT_ADMISSIONS = 500

state_sensitivity = (
    analysis_df
    .groupBy("STFIPS", "Employment_Status")
    .agg(
        F.count("*").alias("admissions"),
        F.sum("Low_Intensity_Placement").alias("low_intensity_admissions")
    )
    .withColumn(
        "low_intensity_placement_rate",
        F.col("low_intensity_admissions") / F.col("admissions")
    )
    .filter(F.col("admissions") >= MIN_STATE_EMPLOYMENT_ADMISSIONS)
)

# Rank employment groups within each state by low-intensity placement rate
state_window = Window.partitionBy("STFIPS").orderBy(F.col("low_intensity_placement_rate").desc())

state_sensitivity_ranked = (
    state_sensitivity
    .withColumn("rank_within_state", F.row_number().over(state_window))
    .orderBy("STFIPS", "rank_within_state")
)

display(state_sensitivity_ranked)

StatementMeta(, 782d0187-be47-4ab6-8924-3e81cedd39aa, 19, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 5f80f117-5134-4bfb-924e-f723a65cc8b6)

## 11.1 State Sensitivity Summary

In [ ]:
state_top_group_summary = (
    state_sensitivity_ranked
    .filter(F.col("rank_within_state") == 1)
    .groupBy("Employment_Status")
    .agg(
        F.count("*").alias("states_ranked_highest")
    )
    .orderBy(F.col("states_ranked_highest").desc())
)

display(state_top_group_summary)

StatementMeta(, 782d0187-be47-4ab6-8924-3e81cedd39aa, 20, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 6f7e3975-31d2-4aa7-bea0-23c2f1682368)

## 11.2 State Sensitivity Coverage

In [ ]:
state_sensitivity_coverage = (
    state_sensitivity_ranked
    .select("STFIPS")
    .distinct()
    .agg(
        F.count("*").alias("states_included_after_min_sample_filter")
    )
)

display(state_sensitivity_coverage)

StatementMeta(, 782d0187-be47-4ab6-8924-3e81cedd39aa, 21, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 753ca09f-ed78-4074-b2c9-66fbb23855e2)

## 12. Interpretation Summary

The statistical validation supports employment status as a meaningful access-context variable for Low-Intensity Placement among admissions with co-occurring mental health needs.

The chi-square test found a statistically significant association between employment status and Low-Intensity Placement. Logistic regression showed that the association remained significant after adjusting for age, sex, race, state, and wait-time category.

Sensitivity checks showed that the employment-based low-intensity placement pattern was not isolated to a single subgroup. Part-time employed admissions ranked highest across all age groups; employed groups ranked highest across all wait-time groups; and employed groups ranked highest in 25 of 37 states with sufficient subgroup sample size.

These results should be interpreted as evidence of a robust descriptive pattern, not causal proof. Additional operational data, such as program schedule, service modality, intake routing rules, insurance coverage, and follow-up outcomes, would be needed to evaluate why this pattern occurs.


## 13. EDA Check: Unknown Employment Values

In [ ]:
from pyspark.sql import functions as F

# Reload source table in case the notebook session was restarted
df = spark.table("silver_tedsa_admissions_cleaned")

# Check unknown / invalid employment values in the co-occurring mental health sample
employment_eda = (
    df
    .filter(F.col("PSYPROB") == 1)
    .withColumn(
        "Employment_Value_Group",
        F.when(F.col("EMPLOY").isin(1, 2, 3, 4), "Usable employment value")
         .otherwise("Unknown / not collected / invalid")
    )
    .groupBy("Employment_Value_Group")
    .agg(
        F.count("*").alias("admissions")
    )
)

employment_total = employment_eda.agg(F.sum("admissions").alias("total_admissions"))

employment_eda_pct = (
    employment_eda
    .crossJoin(employment_total)
    .withColumn(
        "share_of_cooccurring_admissions",
        F.col("admissions") / F.col("total_admissions")
    )
    .orderBy(F.col("admissions").desc())
)

display(employment_eda_pct)

StatementMeta(, b6d6f17f-0735-4c83-a62e-d2cb5e214b44, 5, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 14855ff0-059a-4f04-a759-c1c61700449e)

## 14. EDA Check: Treatment Intensity Distribution

In [ ]:
from pyspark.sql import functions as F

# Reload source table in case the notebook session was restarted
df = spark.table("silver_tedsa_admissions_cleaned")

# Check whether treatment intensity groups are balanced or concentrated
treatment_intensity_eda = (
    df
    .filter(F.col("PSYPROB") == 1)
    .filter(F.col("Treatment_Intensity").isNotNull())
    .groupBy("Treatment_Intensity")
    .agg(
        F.count("*").alias("admissions")
    )
)

treatment_intensity_total = treatment_intensity_eda.agg(F.sum("admissions").alias("total_admissions"))

treatment_intensity_eda_pct = (
    treatment_intensity_eda
    .crossJoin(treatment_intensity_total)
    .withColumn(
        "share_of_cooccurring_admissions",
        F.col("admissions") / F.col("total_admissions")
    )
    .orderBy(F.col("admissions").desc())
)

display(treatment_intensity_eda_pct)

StatementMeta(, b6d6f17f-0735-4c83-a62e-d2cb5e214b44, 6, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, a67f2c28-6cf6-48b2-8f15-741ff148b342)

## 15. EDA Check: Raw Treatment Service Distribution

In [ ]:
from pyspark.sql import functions as F

# Reload source table in case the notebook session was restarted
df = spark.table("silver_tedsa_admissions_cleaned")

# Raw SERVICES distribution before grouping into treatment intensity
services_eda = (
    df
    .filter(F.col("PSYPROB") == 1)
    .groupBy("SERVICES")
    .agg(
        F.count("*").alias("admissions")
    )
)

services_total = services_eda.agg(F.sum("admissions").alias("total_admissions"))

services_eda_pct = (
    services_eda
    .crossJoin(services_total)
    .withColumn(
        "share_of_cooccurring_admissions",
        F.col("admissions") / F.col("total_admissions")
    )
    .orderBy(F.col("admissions").desc())
)

display(services_eda_pct)

StatementMeta(, b6d6f17f-0735-4c83-a62e-d2cb5e214b44, 7, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 55ae6cbe-ce13-4d31-aaa6-9f215e989e3c)